In [1]:
from code_utils import *

In [3]:
# Alfonsi + LS

# Paramters
N = 50    		# Number of exercising dates
N_bar = 5     	# Number of intervals between two exercising dates
K = 100   		# Strike price
r = 0.05  		# Risk-free rate
sigma = 0.3   	# Volatility
T = 0.2   		# Maturity

seed = 42    	# Seed for reproducibility
seed_weights = 12345  # Seed for the random weights

nneurons_mart = 9 # Number of neurons for computing the martingale 
nneurons_reg = 5 # Number of neurons for computing the regression basis
sigma_rnn_mart = sigma # Standard deviation for the random weights
sigma_rnn_reg = sigma # Standard deviation for the random weights 

paths_alfonsi = 10**5
paths_ls_train = 10**5
paths_ls_pricing = 10**5

real_prices = np.array([ 40.00000000, 39.00000000, 38.00000000, 37.00000000, 36.00000000,
    35.00000000, 34.00000000, 33.00000000, 32.00000000, 31.00000000,
    30.00000000, 29.00000000, 28.00000000, 27.00000000, 26.00000000,
    25.00000000, 24.00000000, 23.00000000, 22.00000000, 21.00000000,
    20.00070685, 19.01347004, 18.04313091, 17.09109638, 16.15883366,
    15.24783858, 14.35960625, 13.49560169, 12.65723381, 11.84582143,
    11.06257234, 10.30856320,  9.58471980,  8.89180771,  8.23041673,
     7.60094690,  7.00361025,  6.43844700,  5.90532237,  5.40390521,
     4.93370425,  4.49410547,  4.08432251,  3.70344748,  3.35050142,
     3.02435964,  2.72386986,  2.44779995,  2.19486933,  1.96380311,
     1.75326609,  1.56197956,  1.38862261,  1.23195119,  1.09070376,
     0.96370545,  0.84978944,  0.74786820,  0.65689777,  0.57588855,
     0.50392803,  0.44013960,  0.38373063,  0.33395543,  0.29012326,
     0.25161274,  0.21784490,  0.18829301,  0.16248633,  0.13999277,
     0.12042199,  0.10342780,  0.08869869,  0.07595400,  0.06494518,
     0.05545341,  0.04728321,  0.04026147,  0.03423616,  0.02907429,
     0.02465918
])

prices = np.empty(81)
uncertainty = np.empty(81)
percentage_error = np.empty(81)

for S0 in range(60,141):

    # Temporal grids
    exercise_time = np.linspace(0, T, N + 1)
    exercise_indices = range(0, N_bar * N + 1, N_bar)
    dense_grid = np.linspace(0, T, N_bar * N + 1)

    discount_factor = np.exp(-r * dense_grid)
    discount_factor_exercise_time = np.exp(-r * exercise_time)

    # Step 1 : determining the martingale for the control variate using Alfonsi's scheme
    asset_alfonsi = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths_alfonsi, seed)
    discounted_asset_alfonsi = asset_alfonsi * discount_factor[:, np.newaxis]
    discounted_payoff_alfonsi= (np.maximum(K - asset_alfonsi[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])
    
    coefficients_alfonsi, _ , weights_mart, stats_mart = computing_coefficients_martingale_alfonsi_rnn(N, N_bar, nneurons_mart, discounted_asset_alfonsi, discounted_payoff_alfonsi, paths_alfonsi, asset_alfonsi, sigma_rnn_mart, seed_weights)

    # Step 2 : Longstaff-Schwartz method to get the optimal stopping rule
    asset_ls = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths_ls_train, seed + 1)
    discounted_asset_ls = asset_ls * discount_factor[:, np.newaxis]

    stats_reg  = fit_log_standardizer(asset_ls)
    ls_log = apply_log_standardizer(asset_ls, stats_reg)
    ls_log_ex = ls_log[list(exercise_indices), :]      # (N+1, paths)
    
    basis_ls = np.ones((N-1, paths_ls_train, nneurons_reg + 1))
    basis_ls_mod, weights_reg = random_basis(ls_log_ex[1:N,:], nneurons_reg, sigma_rnn_reg, seed_weights + 1)
    basis_ls = basis_ls_mod.transpose(1, 2, 0)


    regression_coefs, train_deg = ls_train_alfonsi_rnn(K, N, N_bar, nneurons_mart, asset_ls, discounted_asset_ls, basis_ls, discount_factor_exercise_time, coefficients_alfonsi, paths_ls_train, exercise_indices, nneurons_reg, sigma_rnn_mart, weights_mart, stats_mart, seed_weights)
    # Step 3 : Pricing the option using the optimal stopping rule and the martingale as control variate
    asset_pricing = S0 * simulate_gbm(r, sigma, T, dense_grid[1] - dense_grid[0], paths_ls_pricing, seed + 2)
    discounted_asset_pricing = asset_pricing * discount_factor[:, np.newaxis]
    discounted_payoff_pricing = (np.maximum(K - asset_pricing[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])

    ls_price = apply_log_standardizer(asset_pricing, stats_reg)  
    ls_price_ex = ls_price[list(exercise_indices), :]
    basis_pricing = np.ones((N-1, paths_ls_pricing, nneurons_reg + 1))
    basis_pricing_mod, _ = random_basis(ls_price_ex[1:N,:], nneurons_reg, sigma_rnn_reg, seed_weights, weights=weights_reg)
    basis_pricing = basis_pricing_mod.transpose(1, 2, 0)

        
    optimal_stopping, martingale_pricing = ls_apply_alfonsi_rnn( K, N, N_bar, nneurons_mart, asset_pricing, discounted_asset_pricing, basis_pricing, discount_factor_exercise_time, coefficients_alfonsi, regression_coefs, paths_ls_pricing, exercise_indices, train_deg, sigma_rnn_mart, weights_mart, stats_mart, seed_weights)

    # Determining the coefficient for the control variate
    payoff_control_variate = np.sum(discounted_payoff_pricing[1:] * optimal_stopping,axis=0) # Payoff of the control variate using the optimal stopping rule
    martingale_control_variate = np.sum(martingale_pricing * optimal_stopping,axis=0) # Martingale control variate using the optimal stopping rule
    coeff_lambda = np.sum(payoff_control_variate * martingale_control_variate) / np.sum(martingale_control_variate**2) # Coefficient for the control variate

    # Antithetic variates ( antithetic variates only for the pricing part! )
    modified_payoff = payoff_control_variate - coeff_lambda * martingale_control_variate
    antithetic_sum = (modified_payoff[:paths_ls_pricing//2] + modified_payoff[paths_ls_pricing//2:]) / 2
    price_mc = np.mean(antithetic_sum)
    std_mc = np.std(antithetic_sum, ddof=1) / np.sqrt(paths_ls_pricing//2)
    
    # No antithetic variates version
    #price_mc  = np.mean(payoff_control_variate - coeff_lambda * martingale_control_variate)
    #std_mc = np.std(payoff_control_variate - coeff_lambda * martingale_control_variate, ddof=1) / np.sqrt(paths_ls_pricing)
    

    # Since we are generating the paths using a variance reduction technique (antithetic variates), we need to take
    # this into account: we need to do the average of the "antithetic values", hence this is wrong option_price = np.mean(exercise_payoff)
    # antithetic_sum = (exercise_payoff[:npaths_pricing//2] + exercise_payoff[npaths_pricing//2:]) / 2
    # option_price = np.mean(antithetic_sum)

    immediate_exercise = max(K - S0, 0)
    standard_error = std_mc
    price = price_mc

    if immediate_exercise >= price_mc:
        price = immediate_exercise
        standard_error = 0.0       
    else:
        price = price_mc
        standard_error = std_mc
    
    # Results
    prices[S0-60] = price
    uncertainty[S0-60] = standard_error
    percentage_error[S0-60] = 100 * abs(price - real_prices[S0-60]) / real_prices[S0-60]
    print(f"S0= {S0}, Estimated Price: {prices[S0-60]:.5f}, Real Price: {real_prices[S0-60]:.5f}, Percentage Error: {percentage_error[S0-60]:.2f}%, Uncertainty: {uncertainty[S0-60]:.5f}")

S0= 60, Estimated Price: 40.00000, Real Price: 40.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 61, Estimated Price: 39.00000, Real Price: 39.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 62, Estimated Price: 38.00000, Real Price: 38.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 63, Estimated Price: 37.00000, Real Price: 37.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 64, Estimated Price: 36.00000, Real Price: 36.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 65, Estimated Price: 35.00000, Real Price: 35.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 66, Estimated Price: 34.00000, Real Price: 34.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 67, Estimated Price: 33.00000, Real Price: 33.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 68, Estimated Price: 32.00000, Real Price: 32.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 69, Estimated Price: 31.00000, Real Price: 31.00000, Percentage Error

In [4]:
np.savez('rnn_lower_bound_alfonsi_results.npz', S0=np.arange(60,141), price=prices, error=percentage_error, uncertainty=uncertainty)

In [5]:
def rnn_lower_bound_american_put_alfonsi(K, S0, N, N_bar, r, sigma, T, seed, paths_alfonsi, paths_ls_train, paths_ls_pricing, nneurons_mart, nneurons_reg, sigma_rnn_mart, sigma_rnn_reg, seed_weights):
    # Temporal grids
    exercise_time = np.linspace(0, T, N + 1)
    exercise_indices = range(0, N_bar * N + 1, N_bar)
    dense_grid = np.linspace(0, T, N_bar * N + 1)

    discount_factor = np.exp(-r * dense_grid)
    discount_factor_exercise_time = np.exp(-r * exercise_time)

    # Step 1 : determining the martingale for the control variate using Alfonsi's scheme
    asset_alfonsi = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths_alfonsi, seed)
    discounted_asset_alfonsi = asset_alfonsi * discount_factor[:, np.newaxis]
    discounted_payoff_alfonsi= (np.maximum(K - asset_alfonsi[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])
    
    coefficients_alfonsi, _ , weights_mart, stats_mart = computing_coefficients_martingale_alfonsi_rnn(N, N_bar, nneurons_mart, discounted_asset_alfonsi, discounted_payoff_alfonsi, paths_alfonsi, asset_alfonsi, sigma_rnn_mart, seed_weights)

    # Step 2 : Longstaff-Schwartz method to get the optimal stopping rule
    asset_ls = S0 * simulate_gbm_nv(r, sigma, T, dense_grid[1] - dense_grid[0], paths_ls_train, seed + 1)
    discounted_asset_ls = asset_ls * discount_factor[:, np.newaxis]

    stats_reg  = fit_log_standardizer(asset_ls)
    ls_log = apply_log_standardizer(asset_ls, stats_reg)
    ls_log_ex = ls_log[list(exercise_indices), :]      # (N+1, paths)
    
    basis_ls = np.ones((N-1, paths_ls_train, nneurons_reg + 1))
    basis_ls_mod, weights_reg = random_basis(ls_log_ex[1:N,:], nneurons_reg, sigma_rnn_reg, seed_weights + 1)
    basis_ls = basis_ls_mod.transpose(1, 2, 0)


    regression_coefs, train_deg = ls_train_alfonsi_rnn(K, N, N_bar, nneurons_mart, asset_ls, discounted_asset_ls, basis_ls, discount_factor_exercise_time, coefficients_alfonsi, paths_ls_train, exercise_indices, nneurons_reg, sigma_rnn_mart, weights_mart, stats_mart, seed_weights)
    # Step 3 : Pricing the option using the optimal stopping rule and the martingale as control variate
    asset_pricing = S0 * simulate_gbm(r, sigma, T, dense_grid[1] - dense_grid[0], paths_ls_pricing, seed + 2)
    discounted_asset_pricing = asset_pricing * discount_factor[:, np.newaxis]
    discounted_payoff_pricing = (np.maximum(K - asset_pricing[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])

    ls_price = apply_log_standardizer(asset_pricing, stats_reg)  
    ls_price_ex = ls_price[list(exercise_indices), :]
    basis_pricing = np.ones((N-1, paths_ls_pricing, nneurons_reg + 1))
    basis_pricing_mod, _ = random_basis(ls_price_ex[1:N,:], nneurons_reg, sigma_rnn_reg, seed_weights, weights=weights_reg)
    basis_pricing = basis_pricing_mod.transpose(1, 2, 0)

        
    optimal_stopping, martingale_pricing = ls_apply_alfonsi_rnn( K, N, N_bar, nneurons_mart, asset_pricing, discounted_asset_pricing, basis_pricing, discount_factor_exercise_time, coefficients_alfonsi, regression_coefs, paths_ls_pricing, exercise_indices, train_deg, sigma_rnn_mart, weights_mart, stats_mart, seed_weights)

    # Determining the coefficient for the control variate
    payoff_control_variate = np.sum(discounted_payoff_pricing[1:] * optimal_stopping,axis=0) # Payoff of the control variate using the optimal stopping rule
    martingale_control_variate = np.sum(martingale_pricing * optimal_stopping,axis=0) # Martingale control variate using the optimal stopping rule
    coeff_lambda = np.sum(payoff_control_variate * martingale_control_variate) / np.sum(martingale_control_variate**2) # Coefficient for the control variate

    # Antithetic variates ( antithetic variates only for the pricing part! )
    modified_payoff = payoff_control_variate - coeff_lambda * martingale_control_variate
    antithetic_sum = (modified_payoff[:paths_ls_pricing//2] + modified_payoff[paths_ls_pricing//2:]) / 2
    price_mc = np.mean(antithetic_sum)
    std_mc = np.std(antithetic_sum, ddof=1) / np.sqrt(paths_ls_pricing//2)
    
    # No antithetic variates version
    #price_mc  = np.mean(payoff_control_variate - coeff_lambda * martingale_control_variate)
    #std_mc = np.std(payoff_control_variate - coeff_lambda * martingale_control_variate, ddof=1) / np.sqrt(paths_ls_pricing)
    

    # Since we are generating the paths using a variance reduction technique (antithetic variates), we need to take
    # this into account: we need to do the average of the "antithetic values", hence this is wrong option_price = np.mean(exercise_payoff)
    # antithetic_sum = (exercise_payoff[:npaths_pricing//2] + exercise_payoff[npaths_pricing//2:]) / 2
    # option_price = np.mean(antithetic_sum)

    immediate_exercise = max(K - S0, 0)
    standard_error = std_mc
    price = price_mc

    if immediate_exercise >= price_mc:
        price = immediate_exercise
        standard_error = 0.0       
    else:
        price = price_mc
        standard_error = std_mc
        
    return price, standard_error
    

In [6]:
# Alfonsi + LS

# Paramters
N = 50    		# Number of exercising dates
N_bar = 5     	# Number of intervals between two exercising dates
K = 100   		# Strike price
r = 0.05  		# Risk-free rate
sigma = 0.3   	# Volatility
T = 0.2   		# Maturity

seed = 42    	# Seed for reproducibility
seed_weights = 12345  # Seed for the random weights

nneurons_mart = 9 # Number of neurons for computing the martingale 
nneurons_reg = 5 # Number of neurons for computing the regression basis
sigma_rnn_mart = sigma # Standard deviation for the random weights
sigma_rnn_reg = sigma # Standard deviation for the random weights 

paths_alfonsi = 10**5
paths_ls_train = 10**5
paths_ls_pricing = 10**5

real_prices = np.array([ 40.00000000, 39.00000000, 38.00000000, 37.00000000, 36.00000000,
    35.00000000, 34.00000000, 33.00000000, 32.00000000, 31.00000000,
    30.00000000, 29.00000000, 28.00000000, 27.00000000, 26.00000000,
    25.00000000, 24.00000000, 23.00000000, 22.00000000, 21.00000000,
    20.00070685, 19.01347004, 18.04313091, 17.09109638, 16.15883366,
    15.24783858, 14.35960625, 13.49560169, 12.65723381, 11.84582143,
    11.06257234, 10.30856320,  9.58471980,  8.89180771,  8.23041673,
     7.60094690,  7.00361025,  6.43844700,  5.90532237,  5.40390521,
     4.93370425,  4.49410547,  4.08432251,  3.70344748,  3.35050142,
     3.02435964,  2.72386986,  2.44779995,  2.19486933,  1.96380311,
     1.75326609,  1.56197956,  1.38862261,  1.23195119,  1.09070376,
     0.96370545,  0.84978944,  0.74786820,  0.65689777,  0.57588855,
     0.50392803,  0.44013960,  0.38373063,  0.33395543,  0.29012326,
     0.25161274,  0.21784490,  0.18829301,  0.16248633,  0.13999277,
     0.12042199,  0.10342780,  0.08869869,  0.07595400,  0.06494518,
     0.05545341,  0.04728321,  0.04026147,  0.03423616,  0.02907429,
     0.02465918
])


for S0 in range(60,141):
    price, standard_error = rnn_lower_bound_american_put_alfonsi(K, S0, N, N_bar, r, sigma, T, seed, paths_alfonsi, paths_ls_train, paths_ls_pricing, nneurons_mart, nneurons_reg, sigma_rnn_mart, sigma_rnn_reg, seed_weights)
    percentage_error = 100 * abs(price - real_prices[S0-60]) / real_prices[S0-60]
    print(f"S0= {S0}, Estimated Price: {price:.5f}, Real Price: {real_prices[S0-60]:.5f}, Percentage Error: {percentage_error:.2f}%, Uncertainty: {standard_error:.5f}")
    

S0= 60, Estimated Price: 40.00000, Real Price: 40.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 61, Estimated Price: 39.00000, Real Price: 39.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 62, Estimated Price: 38.00000, Real Price: 38.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 63, Estimated Price: 37.00000, Real Price: 37.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 64, Estimated Price: 36.00000, Real Price: 36.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 65, Estimated Price: 35.00000, Real Price: 35.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 66, Estimated Price: 34.00000, Real Price: 34.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 67, Estimated Price: 33.00000, Real Price: 33.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 68, Estimated Price: 32.00000, Real Price: 32.00000, Percentage Error: 0.00%, Uncertainty: 0.00000
S0= 69, Estimated Price: 31.00000, Real Price: 31.00000, Percentage Error